# The Proper EDA: Lending Club Loan Data (2007-2018)

A deep exploratory analysis of 2.26 million consumer loans originated through Lending Club. This notebook treats EDA as the product - not a quick pass before modeling, but a rigorous, structured investigation of data quality, distributional properties, temporal dynamics, multivariate structure, and portfolio-level credit analytics.

**Dataset:** [Lending Club Loan Data](https://www.kaggle.com/datasets/wordsforthewise/lending-club) - 2.26M loans, 151 features, 2007-2018.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster, leaves_list
from scipy.spatial.distance import squareform
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:,.2f}".format)

In [ ]:
# -- Color Palette --
C = {
    "primary":    "#1e293b",
    "secondary":  "#334155",
    "accent":     "#2563eb",
    "accent2":    "#7c3aed",
    "risk":       "#dc2626",
    "safe":       "#059669",
    "warn":       "#d97706",
    "light_gray": "#f1f5f9",
    "mid_gray":   "#94a3b8",
    "dark_gray":  "#475569",
    "bg":         "#ffffff",
}

# Grade color scale (A=safest green through G=riskiest red)
GRADE_COLORS = {
    "A": "#059669", "B": "#10b981", "C": "#d97706",
    "D": "#f59e0b", "E": "#ef4444", "F": "#dc2626", "G": "#991b1b",
}

# Sequential blue palette for heatmaps
BLUE_CMAP = LinearSegmentedColormap.from_list("blue_seq", ["#f0f9ff", "#2563eb", "#1e3a5f"])
# Diverging palette
DIV_CMAP = LinearSegmentedColormap.from_list("div", ["#059669", "#f1f5f9", "#dc2626"])

# -- Matplotlib Style --
plt.rcParams.update({
    "figure.facecolor": "#ffffff",
    "axes.facecolor": "#ffffff",
    "axes.edgecolor": "#e2e8f0",
    "axes.labelcolor": C["primary"],
    "axes.titlecolor": C["primary"],
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "axes.grid": True,
    "grid.color": "#f1f5f9",
    "grid.linewidth": 0.8,
    "xtick.color": C["dark_gray"],
    "ytick.color": C["dark_gray"],
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "font.family": "sans-serif",
    "font.sans-serif": ["Segoe UI", "Helvetica Neue", "Arial"],
    "figure.dpi": 100,
    "savefig.dpi": 150,
    "legend.frameon": False,
    "legend.fontsize": 9,
})

def style_axis(ax, title="", subtitle="", xlabel="", ylabel=""):
    """Apply consistent styling to a matplotlib axis."""
    if title:
        ax.set_title(title, fontsize=14, fontweight="bold", color=C["primary"], pad=12, loc="left")
    if subtitle:
        ax.text(0, 1.02, subtitle, transform=ax.transAxes, fontsize=10,
                color=C["dark_gray"], style="italic", va="bottom")
    if xlabel:
        ax.set_xlabel(xlabel, fontsize=11, color=C["secondary"])
    if ylabel:
        ax.set_ylabel(ylabel, fontsize=11, color=C["secondary"])
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color("#e2e8f0")
    ax.spines["bottom"].set_color("#e2e8f0")
    return ax

def fmt_pct(x, _=None):
    return f"{x:.0%}" if abs(x) < 10 else f"{x:.1%}"

def fmt_k(x, _=None):
    if abs(x) >= 1e6:
        return f"{x/1e6:.1f}M"
    if abs(x) >= 1e3:
        return f"{x/1e3:.0f}K"
    return f"{x:.0f}"

print("Style configured.")

In [ ]:
df = pd.read_csv("data/accepted_2007_to_2018Q4.csv.gz", low_memory=False)

# Parse dates
df["issue_d"] = pd.to_datetime(df["issue_d"], format="mixed")
df["earliest_cr_line"] = pd.to_datetime(df["earliest_cr_line"], format="mixed", errors="coerce")

# Derive key columns upfront
df["issue_year"] = df["issue_d"].dt.year
df["issue_month"] = df["issue_d"].dt.to_period("M")
df["credit_history_years"] = (df["issue_d"] - df["earliest_cr_line"]).dt.days / 365.25

# Binary default indicator (Charged Off or Default = 1, else 0)
default_statuses = ["Charged Off", "Default"]
df["is_default"] = df["loan_status"].isin(default_statuses).astype(int)

print(f"Loaded: {df.shape[0]:,} loans x {df.shape[1]} features")
print(f"Date range: {df['issue_d'].min():%Y-%m} to {df['issue_d'].max():%Y-%m}")
print(f"Default rate: {df['is_default'].mean():.2%}")